In [2]:
import sys
import os
import re
import csv
from decimal import Decimal

from vcd_parser import *

In [3]:
# Load VCD file
vcd_file = "/home/jianliu/TCAD25/dynamatic-mlir/integration-test/fir/out/sim/HLS_VERIFY/trace.vcd"
TCP = 8

In [ ]:

node_types = set((
    "cmpi",
    "addi",
    "subi",
    "muli",
    "extsi",
    "buffer",
    "mc_load",
    "mc_store",
    "lsq_load",
    "lsq_store",
    "merge",
    "control_merge",
    "fork",
    "d_return",
    "cond_br",
    "end",
    "andi",
    "ori",
    "xori",
    "shli",
    "shrsi"
    "shrui",
    "select",
    "mux",
    "source",
    "sink",
    "trunci",
    "constant",
    "extui"
))

In [5]:
# Parse the vcd file
vcd_file = VcdParser(vcd_file)
coefficient = Decimal('1e-9')/vcd_file.timescale_dict["factor"]

In [6]:
###############################################
####### Function Definition
###############################################
def extract_node_names(signals, node_type_set):
    matched_signals = set()

    # Iterate over each signal
    for signal in signals:
        # Extract the signal names directly under the top wrapper
        wrapper_signal_name = signal.split(".")[-1]
        pattern = re.compile(r'(' + '|'.join(re.escape(nt) for nt in node_types) + r')(\d+)')
        match = re.search(pattern, wrapper_signal_name)

        if match:
            matched_signals.add(match.group(1)+match.group(2))
            
    return sorted(list(matched_signals))   

In [7]:
# Extract complete node list from the vcd file
node_list = extract_node_names(vcd_file.unique_signal_names, node_types)
node_switching = {}

real_clock_cycles = vcd_file.endtime / (coefficient * TCP)

In [ ]:
# Update all node switching information
for sel_node in node_list:
    # Update the switching information
    vcd_file.get_node_toggle_count(sel_node)

    if sel_node not in node_switching:
        node_switching[sel_node] = []

    # Data Channel Switching
    node_switching[sel_node].append(vcd_file.node_switching_info[sel_node].total_dataout_channel_switching)
    
    # Handshake Valid Channel Switching
    node_switching[sel_node].append(vcd_file.node_switching_info[sel_node].total_valid_switching)
    
    # Handshake Ready Channel Switching
    node_switching[sel_node].append(vcd_file.node_switching_info[sel_node].total_ready_switching)

    